## 1. MLP 방식 (learnX MLP)

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# MLP 정의 (고차원 벡터 -> 저차원 벡터)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)  # 고차원 -> 은닉층
        self.layer2 = nn.Linear(hidden_dim, output_dim)  # 은닉층 -> 저차원
        
    def forward(self, x):
        x = torch.relu(self.layer1(x))  # 활성화 함수 ReLU
        x = self.layer2(x)  # 출력층
        return x

# 폴더 내의 모든 Feather 파일에 대해 수행
input_folder = '../output/concatenate/concatenate(BERT ver.)(UI+UR)+norm)'  # 입력 폴더 경로
output_folder = '../output/lowdim_vectors/lowdim_vectors(BERT ver.)(UI+UR)'  # 출력 폴더 경로

# 폴더가 존재하지 않으면 생성
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# MLP 모델 초기화 (예시: 벡터 차원 1000 -> 50)
hidden_dim = 128  # 은닉층 차원
output_dim = 64   # 저차원 벡터 크기

# 건너뛰어진 파일을 기록할 리스트
skipped_files = []

# 폴더 내 모든 feather 파일 처리
for filename in os.listdir(input_folder):
    if filename.endswith('.feather'):
        input_file = os.path.join(input_folder, filename)
        print(f"Processing file: {input_file}")
        
        try:
            # Feather 파일 읽기
            df = pd.read_feather(input_file)
            
            # 각 파일마다 'concatenated_vector' 열의 첫 번째 행을 통해 input_dim 설정
            sample_vector = df['concatenated_vector'].iloc[10]  # 첫 번째 행 벡터 가져오기
            input_dim = len(sample_vector)  # 고차원 벡터 크기
            
            # MLP 모델 초기화
            model = MLP(input_dim, hidden_dim, output_dim)

            # 모델을 평가 모드로 전환
            model.eval()

            # 결과를 저장할 리스트
            output_data = []

            # 각 사용자에 대해 벡터 압축
            for idx, row in df.iterrows():
                vector = row['concatenated_vector']
                # 첫 번째 행의 길이와 일치하는지 확인
                if len(vector) != input_dim:
                    print(f"Skipping row {idx} due to vector length mismatch: expected {input_dim}, got {len(vector)}")
                    continue
                vector_tensor = torch.tensor(vector, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad():
                    reduced_vector = model(vector_tensor)
                output_data.append({
                    'UserID': row['UserID'], 
                    'lowdim_vector': reduced_vector.squeeze().numpy().tolist()
                })

            # 결과를 DataFrame으로 변환
            output_df = pd.DataFrame(output_data)

            # 출력 파일 경로 설정
            output_filename = f"{os.path.splitext(filename)[0]}_lowdim.feather"
            output_file = os.path.join(output_folder, output_filename)

            # 결과를 새로운 feather 파일로 저장
            output_df.to_feather(output_file)

            print(f"Saved reduced vector to {output_file}")

        except Exception as e:
            # 오류가 발생한 파일을 skipped_files 리스트에 추가
            skipped_files.append((filename, str(e)))
            print(f"Skipping file {filename} due to error: {e}")

# 처리되지 않은 파일 출력
if skipped_files:
    print("\nThe following files were skipped due to errors:")
    for file, error in skipped_files:
        print(f"{file}: {error}")

print("Processing completed for all files.")


## 2. MLP 기반 Autoencoder (학습 + 변환 전체 코드)

Autoencoder로 학습 후 인코더에서 나온 저차원 벡터만 저장

* Autoencoder: 단순한 3층 구조, 입력 → 은닉 → 압축 → 은닉 → 복원

* 학습 대상: 입력 벡터 자체 (입력 = 타겟)

* 결과: encoder의 출력만 사용해서 차원 축소된 벡터로 저장



In [28]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# 경로 설정
input_folder = '../output/concatenate/concatenate(BERT ver.)(UI+UR)+norm'
output_folder = '../output/lowdim_vectors/lowdim_vectors(BERT ver.)(UI+UR)+norm+256dim'
os.makedirs(output_folder, exist_ok=True)

# 설정
hidden_dim = 512
output_dim = 256
epochs = 10
batch_size = 32
lr = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Autoencoder 정의
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

skipped_files = []

# Feather 파일 처리
for filename in os.listdir(input_folder):
    if filename.endswith('.feather'):
        input_path = os.path.join(input_folder, filename)
        print(f"\n🔵 Processing: {filename}")

        try:
            df = pd.read_feather(input_path)
            vectors = df['concatenated_vector'].tolist()
            sample_vector = vectors[0]
            input_dim = len(sample_vector)

            # 벡터 길이 일치 확인
            filtered_vectors = [v for v in vectors if len(v) == input_dim]
            if len(filtered_vectors) < len(vectors):
                print(f" - Skipping {len(vectors) - len(filtered_vectors)} rows due to dim mismatch")

            # 텐서 변환
            all_tensor = torch.tensor(filtered_vectors, dtype=torch.float32).to(device)
            dataset = TensorDataset(all_tensor)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            # Autoencoder 초기화 및 학습
            model = Autoencoder(input_dim, hidden_dim, output_dim).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            criterion = nn.MSELoss()

            model.train()
            for epoch in range(epochs):
                total_loss = 0
                for batch in loader:
                    batch_data = batch[0]
                    optimizer.zero_grad()
                    encoded, decoded = model(batch_data)
                    loss = criterion(decoded, batch_data)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                print(f"   Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

            # 인코딩 후 저장
            model.eval()
            output_data = []
            for idx, row in df.iterrows():
                vector = row['concatenated_vector']
                if len(vector) != input_dim:
                    continue
                vec_tensor = torch.tensor(vector, dtype=torch.float32).unsqueeze(0).to(device)
                with torch.no_grad():
                    encoded, _ = model(vec_tensor)
                output_data.append({
                    'UserID': row['UserID'],
                    'lowdim_vector': encoded.squeeze().cpu().numpy().tolist()
                })

            output_df = pd.DataFrame(output_data)
            output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.feather")
            output_df.to_feather(output_path)
            print(f"✅ Saved: {output_path}")
        
        except Exception as e:
            print(f" - ❌ Skipped due to error: {e}")
            skipped_files.append((filename, str(e)))

# 스킵된 파일 출력
if skipped_files:
    print("\n🔴 Skipped files:")
    for fname, reason in skipped_files:
        print(f" - {fname}: {reason}")
else:
    print("\n✅ All files processed successfully.")



🔵 Processing: Adult Products.feather
   Epoch 1/10, Loss: 0.0310
   Epoch 2/10, Loss: 0.0275
   Epoch 3/10, Loss: 0.0232
   Epoch 4/10, Loss: 0.0194
   Epoch 5/10, Loss: 0.0130
   Epoch 6/10, Loss: 0.0090
   Epoch 7/10, Loss: 0.0066
   Epoch 8/10, Loss: 0.0056
   Epoch 9/10, Loss: 0.0046
   Epoch 10/10, Loss: 0.0039
✅ Saved: ../output/lowdim_vectors/lowdim_vectors(BERT ver.)(UI+UR)+norm+256dim\Adult Products.feather

🔵 Processing: Beauty.feather
   Epoch 1/10, Loss: 0.0977
   Epoch 2/10, Loss: 0.0253
   Epoch 3/10, Loss: 0.0176
   Epoch 4/10, Loss: 0.0127
   Epoch 5/10, Loss: 0.0099
   Epoch 6/10, Loss: 0.0072
   Epoch 7/10, Loss: 0.0065
   Epoch 8/10, Loss: 0.0063
   Epoch 9/10, Loss: 0.0059
   Epoch 10/10, Loss: 0.0051
✅ Saved: ../output/lowdim_vectors/lowdim_vectors(BERT ver.)(UI+UR)+norm+256dim\Beauty.feather

🔵 Processing: Books.feather
   Epoch 1/10, Loss: 0.0932
   Epoch 2/10, Loss: 0.0315
   Epoch 3/10, Loss: 0.0201
   Epoch 4/10, Loss: 0.0124
   Epoch 5/10, Loss: 0.0105
   Ep

KeyboardInterrupt: 

## MLP (마지막 은닉층)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
from tqdm import tqdm

# ========== ✅ 설정 파라미터 ==========
config = {
    "hidden_dims": [64, 32],
    "output_dim": 16,
    "learning_rate": 0.005,
    "epochs": 100,
    "batch_size": 500,
    "seed": 42
}

# ========== 1. 신뢰 정보 로드 ==========
trust_df = pd.read_csv("../dataset/trustnetwork.csv")
trust_dict = {}
for _, row in trust_df.iterrows():
    uid = int(row["userid"])
    trustors = [int(x) for x in row["trustors"].split(",")]
    trust_dict[uid] = trustors

# ========== 2. 벡터 로드 ==========
concat_df = pd.read_feather("../output/concatenate/concatenate(doc2vec)(2)/Games.feather")
concat_df["UserID"] = concat_df["UserID"].astype(int)
user_embeddings = {int(row["UserID"]): np.array(row["concatenated_vector"]) for _, row in concat_df.iterrows()}
all_user_ids = list(user_embeddings.keys())

# ========== 3. 신뢰/부정 쌍 생성 ==========
positive_pairs = [(user, trusted) for user in trust_dict for trusted in trust_dict[user]
                  if user in user_embeddings and trusted in user_embeddings]

negative_pairs = []
rng = np.random.default_rng(seed=config["seed"])
neg_count = 4 * len(positive_pairs)
while len(negative_pairs) < neg_count:
    u1, u2 = rng.choice(all_user_ids, 2, replace=False)
    if not (u2 in trust_dict.get(u1, []) or u1 in trust_dict.get(u2, [])):
        negative_pairs.append((u1, u2))

# ========== 4. 모델 정의 ==========
class TrustEmbedder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(dims[-1], output_dim))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                insize, outsize = m.weight.shape
                bound = np.sqrt(6 / (insize + outsize))
                nn.init.uniform_(m.weight, -bound, bound)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

def trust_score(u_emb, v_emb):
    return torch.cosine_similarity(u_emb, v_emb)

# ========== 5. 학습 준비 ==========
input_dim = len(next(iter(user_embeddings.values())))
model = TrustEmbedder(input_dim, config["hidden_dims"], config["output_dim"])
optimizer = optim.SGD(model.parameters(), lr=config["learning_rate"])
loss_fn = nn.MSELoss()

pairs = positive_pairs + negative_pairs
labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)

# 최고 성능 저장용
best_f1 = 0
best_model_state = None

# ========== 6. 학습 ==========
for epoch in range(config["epochs"]):
    model.train()
    total_loss = 0
    y_true, y_pred = [], []

    indices = np.arange(len(pairs))
    np.random.shuffle(indices)

    for i in tqdm(range(0, len(pairs), config["batch_size"]), desc=f"Epoch {epoch+1}"):
        batch_idx = indices[i:i + config["batch_size"]]
        batch_pairs = [pairs[j] for j in batch_idx]
        batch_labels = [labels[j] for j in batch_idx]

        batch_loss = 0
        optimizer.zero_grad()

        for (u, v), label in zip(batch_pairs, batch_labels):
            u_emb = torch.tensor(user_embeddings[u], dtype=torch.float32).unsqueeze(0)
            v_emb = torch.tensor(user_embeddings[v], dtype=torch.float32).unsqueeze(0)

            u_proj = model(u_emb)
            v_proj = model(v_emb)

            pred_score = trust_score(u_proj, v_proj)
            loss = loss_fn(pred_score, torch.tensor([label], dtype=torch.float32))
            loss.backward()
            batch_loss += loss.item()

            pred_label = int(pred_score.item() >= 0.5)
            y_true.append(label)
            y_pred.append(pred_label)

        optimizer.step()
        total_loss += batch_loss / len(batch_pairs)

    avg_loss = total_loss / len(pairs)
    f1 = f1_score(y_true, y_pred)
    print(f"Epoch {epoch + 1} Loss: {avg_loss:.4f} | F1 Score: {f1:.4f}")

    # 최고 모델 저장
    if f1 > best_f1:
        best_f1 = f1
        best_model_state = model.state_dict()

# ========== 7. 최고 성능 모델 적용 및 벡터 저장 ==========
model.load_state_dict(best_model_state)
model.eval()

user_lowdim = []
user_ids = []

with torch.no_grad():
    for uid, vec in user_embeddings.items():
        inp = torch.tensor(vec, dtype=torch.float32).unsqueeze(0)
        lowdim = model(inp).squeeze(0).numpy()
        user_ids.append(uid)
        user_lowdim.append(lowdim)

lowdim_df = pd.DataFrame({
    "UserID": user_ids,
    "lowdim_vector": user_lowdim
})
lowdim_df.to_feather("../output/test.feather")
print(f"✅ 저차원 임베딩 저장 완료 (Best F1: {best_f1:.4f})")


Epoch 1: 100%|██████████| 253/253 [01:32<00:00,  2.75it/s]


Epoch 1 Loss: 0.0005 | F1 Score: 0.3358


Epoch 2: 100%|██████████| 253/253 [01:32<00:00,  2.74it/s]


Epoch 2 Loss: 0.0003 | F1 Score: 0.4511


Epoch 3: 100%|██████████| 253/253 [01:36<00:00,  2.62it/s]


...

Epoch 98 Loss: 0.0002 | F1 Score: 0.7487


Epoch 99: 100%|██████████| 253/253 [01:28<00:00,  2.85it/s]


Epoch 99 Loss: 0.0002 | F1 Score: 0.7474


Epoch 100: 100%|██████████| 253/253 [01:28<00:00,  2.85it/s]


Epoch 100 Loss: 0.0002 | F1 Score: 0.7480
✅ 저차원 임베딩 저장 완료 (Best F1: 0.7490)


### 순회

In [29]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# 1. 신뢰 정보 로드 및 파싱
trust_df = pd.read_csv("../dataset/trustnetwork.csv")
trust_dict = {}
for _, row in trust_df.iterrows():
    uid = int(row["userid"])
    trustors = [int(x) for x in row["trustors"].split(",")]
    trust_dict[uid] = trustors

# 2. feather 파일들이 있는 폴더 경로
input_folder = "../output/concatenate/concatenate(doc2vec ver.)"
output_folder = "../output/lowdim_vectors/lowdim_vectors(2)(doc2vec ver.)+128dim"
os.makedirs(output_folder, exist_ok=True)

# 3. feather 파일 순회
for filename in os.listdir(input_folder):
    if not filename.endswith(".feather"):
        continue

    print(f"\n📂 Processing file: {filename}")
    filepath = os.path.join(input_folder, filename)
    
    # 3-1. 사용자 벡터 로드
    concat_df = pd.read_feather(filepath)
    concat_df["UserID"] = concat_df["UserID"].astype(int)

    # 유저 벡터 딕셔너리로 구성
    user_embeddings = {
        int(row["UserID"]): np.array(row["concatenated_vector"]) for _, row in concat_df.iterrows()
    }
    all_user_ids = list(user_embeddings.keys())

    # 4. 신뢰 쌍 T+와 부정 쌍 T− 생성
    positive_pairs = []
    negative_pairs = []

    for user in trust_dict:
        for trusted in trust_dict[user]:
            if trusted in user_embeddings and user in user_embeddings:
                positive_pairs.append((user, trusted))

    rng = np.random.default_rng(seed=42)
    while len(negative_pairs) < len(positive_pairs):
        u1, u2 = rng.choice(all_user_ids, 2, replace=False)
        if not (u2 in trust_dict.get(u1, []) or u1 in trust_dict.get(u2, [])):
            negative_pairs.append((u1, u2))

    # 5. MLP 모델 정의
    class TrustEmbedder(nn.Module):
        def __init__(self, input_dim, output_dim):
            super().__init__()
            self.mlp = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Linear(128, output_dim)
            )

        def forward(self, x):
            return self.mlp(x)

    def trust_score(u_emb, v_emb):
        return torch.cosine_similarity(u_emb, v_emb)

    # 6. 학습 준비
    input_dim = len(next(iter(user_embeddings.values())))
    output_dim = 128

    model = TrustEmbedder(input_dim, output_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # 7. 학습 루프
    epochs = 10
    for epoch in range(epochs):
        total_loss = 0
        model.train()

        pairs = positive_pairs + negative_pairs
        labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)

        for (u, v), label in tqdm(zip(pairs, labels), total=len(pairs), desc=f"[{filename}] Epoch {epoch+1}"):
            u_emb = torch.tensor(user_embeddings[u], dtype=torch.float32).unsqueeze(0)
            v_emb = torch.tensor(user_embeddings[v], dtype=torch.float32).unsqueeze(0)

            u_proj = model(u_emb)
            v_proj = model(v_emb)

            pred_score = trust_score(u_proj, v_proj)
            loss = loss_fn(pred_score, torch.tensor([label], dtype=torch.float32))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(pairs):.4f}")

    # 8. 임베딩 추출 및 저장
    model.eval()
    user_lowdim = []
    user_ids = []

    with torch.no_grad():
        for uid, vec in user_embeddings.items():
            inp = torch.tensor(vec, dtype=torch.float32).unsqueeze(0)
            lowdim = model(inp).squeeze(0).numpy()
            user_ids.append(uid)
            user_lowdim.append(lowdim)

    lowdim_df = pd.DataFrame({
        "UserID": user_ids,
        "lowdim_vector": user_lowdim
    })

    save_path = os.path.join(output_folder, f"{filename}")
    lowdim_df.to_feather(save_path)
    print(f"✅ {save_path} 저장 완료")



📂 Processing file: Adult Products.feather


[Adult Products.feather] Epoch 1: 100%|██████████| 168/168 [00:00<00:00, 534.73it/s]


Epoch 1 Loss: 0.1896


[Adult Products.feather] Epoch 2: 100%|██████████| 168/168 [00:00<00:00, 550.30it/s]


Epoch 2 Loss: 0.1958


...

Epoch 8 Loss: 0.0142


[Beauty.feather] Epoch 9: 100%|██████████| 106330/106330 [04:48<00:00, 368.58it/s]


Epoch 9 Loss: 0.0139


[Beauty.feather] Epoch 10:  14%|█▎        | 14440/106330 [00:39<04:10, 366.65it/s]


KeyboardInterrupt: 

## 확인

In [7]:
import ast  # 문자열을 리스트로 변환하기 위해 사용
file = os.path.join(output_folder, 'Adult Products_lowdim.feather')
df = pd.read_feather(file)
vector = df[df['UserID'] == '11996']['lowdim_vector'].iloc[0]
print("벡터 길이:", len(vector))


NameError: name 'os' is not defined

In [20]:
df = pd.read_feather('../output/test.feather')
df
vector = df[df['UserID'] == 11996]['lowdim_vector'].iloc[0]
print("벡터 길이:", len(vector))

벡터 길이: 32
